# Plot streamlines, PV planview


In [2]:
import wrf
from wrf import (CoordPair, vertcross, get_cartopy, getvar, 
                    latlon_coords, to_np, destagger, ll_to_xy, 
                    smooth2d, interplevel)
import numpy as np
from netCDF4 import Dataset 
import matplotlib.pyplot as plt
import pyart
import geocat.viz as gv
from cartopy.mpl.gridliner import LongitudeFormatter, LatitudeFormatter
import os
import glob
from metpy.units import units
import cartopy.crs as crs
import pandas as pd
import xarray as xr
import metpy 
import metpy.calc as mpcalc
import numpy.ma as ma
import matplotlib.ticker as mticker

import cartopy.feature as cfeature
from metpy.interpolate import cross_section
import pkg_resources

/glade/u/apps/opt/conda/envs/npl-2025b/lib/python3.12/site-packages/wrf/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119



In [3]:
# Set up for saving graphics files
plotsdir = '/glade/u/home/athornton/wrf_visualization/cold_pool_data/dbz_pv_cp/'

# Set directory where wrfout files reside, and list the files for processing.  Set up for a directory with only wrfout files.
os.chdir("/glade/campaign/univ/uncs0067/cold_pool_experiments/coldpool1")
os.getcwd()

'/glade/campaign/univ/uncs0067/cold_pool_experiments/coldpool1'

In [4]:
datafiles = ["wrfout_d03_2018-08-26_00:00:00",
             "wrfout_d03_2018-08-26_03:00:00",
             "wrfout_d03_2018-08-26_09:00:00",
             "wrfout_d03_2018-08-26_12:00:00",
             "wrfout_d03_2018-08-26_14:30:00",
             "wrfout_d03_2018-08-26_18:00:00", 
             "wrfout_d03_2018-08-27_00:00:00", 
]

numfiles=len(datafiles)
print(numfiles)

7


In [5]:
input = "wrfout_d03_2018-08-25_23:30:00" 
ncfile = Dataset(input)
slp = getvar(ncfile, "slp")
lats, lons = latlon_coords(slp)
lats_np = to_np(lats)
lons_np = to_np(lons)
cart_proj = get_cartopy(wrfin=ncfile)
print (cart_proj)
proj=crs.PlateCarree()

+proj=merc +a=6370000.0 +b=6370000.0 +nadgrids=@null +lon_0=0.0 +lat_ts=14.828 +k=1 +units=m +no_defs +type=crs


In [6]:
# Slice these arrays so they work with the new dataset
lat = to_np(lats)[:,0]
lon = to_np(lons)[0]

In [7]:
# define plotting function
def create_fig():
    fig = plt.figure(figsize=(12,12))
    plt.rcParams['savefig.dpi'] = 255

    proj=crs.PlateCarree()

    ax = plt.axes(projection=proj)
    ax.set_extent([-26, -4, 1, 24],crs.PlateCarree())
    #ax.add_feature(cfeature.STATES.with_scale('50m'), linewidth=.5, edgecolor="black")
    ax.add_feature(cfeature.COASTLINE.with_scale('10m'), linewidth=0.8, edgecolor='black')
    ax.add_feature(cfeature.BORDERS.with_scale('50m'), linewidth=1.0)
    
    #gv.set_titles_and_labels(ax, lefttitle=" 600 hPa Zonal Winds", lefttitlefontsize=18)
    plt.title('600-850 streamlines, PV 600', loc='left', weight='bold', fontsize=22)
    plt.title(titletime, loc='right', fontsize=22)
    gl1 = ax.gridlines(crs=proj, linewidth=1, color='darkgray', draw_labels=True)
    gl1.top_labels = False
    gl1.right_labels = False
    gl1.xlocator = mticker.FixedLocator(np.arange(-90,40,4))
    gl1.ylocator = mticker.FixedLocator(np.arange(-2,30,4))
    gl1.xformatter = LongitudeFormatter(degree_symbol='')
    gl1.yformatter = LatitudeFormatter(degree_symbol='')
    gl1.xlabel_style = {'size': 22}
    gl1.ylabel_style = {'size': 22}

    vortlevels = np.arange(-15,15,1)
    cf = ax.contourf(lons_np, lats_np, pv600, 
                          levels=vortlevels, transform=proj,
                          extend='both', cmap='bwr')

    ax.streamplot(lons_np, lats_np, u,v, 
                  density=0.4, broken_streamlines=False, 
                  color='black', linewidth=1.9)
    
    plt.tight_layout()
    cb = plt.colorbar(cf, ax=ax, orientation="horizontal", pad=.06, shrink=.6, aspect=30)
    cb.set_label('PVU', fontsize=16)
    
    outTPlotName= 'pv_streamlines_'+filetime+'.png'
    plt.savefig(plotsdir+outTPlotName)

In [8]:
# times = []
# c_list = []
for i in range(4, 5):
    wrf_out_data = xr.open_dataset(datafiles[i])  
    ncfile = Dataset(datafiles[i])
    Time=wrf.extract_times(ncfile, timeidx=0, method='cat', squeeze=True, cache=None, meta=False, do_xtime=False)
    timestr=(str(Time))
    # Set up one time string for plot titles, another for file names
    titletime=(timestr[0:10]+' '+timestr[11:16])
    filetime=(timestr[0:10]+'_'+timestr[11:16])
    print('WRF valid time: ',filetime)
    
    # Get all the variables we need
    ua = wrf_out_data["U"]
    va = wrf_out_data["V"]
    
    # Unstagger the winds to match the pressure grid
    ua = destagger(ua, stagger_dim=3)             # Unstagger U in the x-direction
    va = destagger(va, stagger_dim=2)             # Unstagger V in the x-direction
    
    # Grid spacing for vorticity calculation
    dx = ncfile.DX * units("m")
    dy = ncfile.DY * units("m")
    
    u_winds = xr.DataArray(ua)[0]
    v_winds = xr.DataArray(va)[0]
    
    u_lay = u_winds.sel(dim_1=slice(12,19))*units('m/s') 
    v_lay = v_winds.sel(dim_1=slice(12,19))*units('m/s')
    
    ### FOR VORTICITY
    # vo = mpcalc.vorticity(u_lay, v_lay, dx=dx, dy=dy).mean(dim='dim_1')
    # vo.name = 'vorticity'
    p = getvar(ncfile, "pressure")
    pv = getvar(ncfile, 'pvo')
    
    pv600 = interplevel(pv, p, 600)    
    
    ### FOR STREAMLINES
    u = u_lay.mean(dim='dim_1')
    v = v_lay.mean(dim='dim_1')
    
    u = to_np(u)
    v = to_np(v)

    f = create_fig()
    plt.close()
    

WRF valid time:  2018-08-26_14:30
